# 08 — Robustness and QC sensitivity checks

**Input:** `results/adata/07_annotated.h5ad`  
**Outputs:** `results/tables/08_*.csv`, `results/figures/08_*.png`, `results/adata/08_robustness.h5ad`

This is a **diagnostic**, not biological validation. Original 29 Leiden labels, normalized expression, counts, spatial graph, and provisional annotations remain intact. **No QC filters are applied.** Alternative resolutions are exploratory and use the **existing expression-neighbor graph**. Spatial homophily is descriptive; random-label permutations do not account for spatial autocorrelation or sampling design.

## 0. Imports and paths

In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse, stats
from sklearn.metrics import adjusted_rand_score

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "results/adata/07_annotated.h5ad").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run 07_biological_annotation first; missing results/adata/07_annotated.h5ad")
INPUT = ROOT / "results/adata/07_annotated.h5ad"
OUTPUT = ROOT / "results/adata/08_robustness.h5ad"
TAB = ROOT / "results/tables"
FIG = ROOT / "results/figures"
for p in (OUTPUT.parent, TAB, FIG): p.mkdir(parents=True, exist_ok=True)
SEED = 42
rng = np.random.default_rng(SEED)
print("Python:", sys.executable)
print("Scanpy:", version("scanpy"), "Project:", ROOT)

## 1. Load and verify data integrity

In [ ]:
adata = sc.read_h5ad(INPUT)
assert adata.shape == (14400, 18085), f"Unexpected dataset shape: {adata.shape}"
assert "leiden" in adata.obs and "spatial" in adata.obsm
assert "counts" in adata.layers and sparse.issparse(adata.layers["counts"])
assert all(k in adata.obsp for k in ("connectivities", "distances", "spatial_connectivities", "spatial_distances"))
assert all(k in adata.obs for k in ("total_counts", "n_genes_by_counts"))
assert "provisional_domain_program" in adata.obs
original_leiden = adata.obs["leiden"].copy()
clusters = list(adata.obs["leiden"].cat.categories)
print("Data:", adata.shape, "Clusters:", len(clusters), "Spatial edges:", adata.obsp["spatial_connectivities"].nnz)

## 2. Cluster size imbalance and QC summaries

Report the number of bins and QC distributions per cluster. **QC association does not prove technical confounding.** The mitochondrial percentage is omitted from correlation testing if it is entirely missing.

In [ ]:
qc_cols = ["total_counts", "n_genes_by_counts"]
if "pct_counts_mt" in adata.obs and pd.to_numeric(adata.obs["pct_counts_mt"], errors="coerce").notna().any():
    qc_cols.append("pct_counts_mt")
obs = adata.obs.copy()
obs["leiden"] = obs["leiden"].astype(str)
for col in qc_cols: obs[col] = pd.to_numeric(obs[col], errors="coerce")
rows = []
for cl, frame in obs.groupby("leiden", sort=False):
    row = {"leiden": cl, "n_bins": len(frame), "fraction": len(frame) / len(obs)}
    for col in qc_cols:
        v = frame[col].dropna()
        row[f"{col}_median"] = float(v.median()) if len(v) else np.nan
        row[f"{col}_q25"] = float(v.quantile(.25)) if len(v) else np.nan
        row[f"{col}_q75"] = float(v.quantile(.75)) if len(v) else np.nan
    rows.append(row)
cluster_qc = pd.DataFrame(rows).sort_values("n_bins", ascending=False)
cluster_qc.to_csv(TAB / "08_cluster_qc_summary.csv", index=False)
print(cluster_qc.head(10).to_string(index=False))
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(cluster_qc["leiden"], cluster_qc["n_bins"])
ax.set(xlabel="Leiden cluster", ylabel="Number of 56 µm bins", title="Cluster size imbalance (original resolution 0.5)")
ax.tick_params(axis="x", rotation=90)
fig.tight_layout(); fig.savefig(FIG / "08_cluster_sizes.png", dpi=170); plt.show()

## 3. QC association with cluster assignments

Kruskal–Wallis tests provide an exploratory assessment of whether QC distributions differ among clusters. Large sample sizes and spatial dependence limit interpretation of p-values; report effect sizes and distributions rather than using p-values alone. For an approximate effect size use `(H - k + 1) / (N - k)`, clipped to [0,1].

In [ ]:
association = []
for col in qc_cols:
    groups = [g[col].dropna().to_numpy(dtype=float) for _, g in obs.groupby("leiden", sort=False)]
    groups = [v for v in groups if len(v)]
    if len(groups) >= 2 and len(np.unique(np.concatenate(groups))) > 1:
        h, p = stats.kruskal(*groups)
        n = sum(map(len, groups)); k = len(groups)
        eps2 = float(np.clip((h - k + 1) / max(1, n-k), 0, 1))
    else: h, p, eps2 = np.nan, np.nan, np.nan
    association.append({"qc_metric": col, "kruskal_H": h, "nominal_p": p, "epsilon_squared_approx": eps2, "note": "exploratory; bins spatially dependent"})
qc_tests = pd.DataFrame(association)
qc_tests.to_csv(TAB / "08_qc_cluster_associations.csv", index=False)
print(qc_tests.to_string(index=False))
fig, axes = plt.subplots(1, len(qc_cols), figsize=(max(8, 5*len(qc_cols)), 5), squeeze=False)
for ax, col in zip(axes[0], qc_cols):
    data = [obs.loc[obs["leiden"] == cl, col].dropna().to_numpy() for cl in clusters]
    ax.boxplot(data, showfliers=False, tick_labels=clusters)
    ax.set(title=col, xlabel="Leiden", ylabel=col)
    ax.tick_params(axis="x", rotation=90, labelsize=7)
fig.suptitle("QC distributions by original cluster (outliers hidden)")
fig.tight_layout(); fig.savefig(FIG / "08_qc_by_cluster.png", dpi=170); plt.show()

## 4. Resolution sensitivity of Leiden clustering

Use the **same expression-neighbor graph** from step 04 and a fixed seed. Recompute labels at resolutions 0.3, 0.5, 0.8 and 1.0. This tests **resolution sensitivity**, not stability to changes in input data, preprocessing, or random seeds. Store alternatives under separate keys and preserve `leiden`.

In [ ]:
RESOLUTIONS = [0.3, 0.5, 0.8, 1.0]
labels = {"original_0.5": original_leiden.astype(str).to_numpy()}
resolution_rows = []
for res in RESOLUTIONS:
    key = "leiden_r" + str(res).replace(".", "p")
    sc.tl.leiden(adata, resolution=res, key_added=key, random_state=SEED, flavor="igraph", directed=False)
    values = adata.obs[key].astype(str).to_numpy()
    labels[key] = values
    resolution_rows.append({"resolution": res, "key": key, "n_clusters": int(adata.obs[key].nunique()), "ARI_vs_original": float(adjusted_rand_score(labels["original_0.5"], values))})
resolution_df = pd.DataFrame(resolution_rows)
resolution_df.to_csv(TAB / "08_resolution_sensitivity.csv", index=False)
print(resolution_df.to_string(index=False))
keys = list(labels)
ari = pd.DataFrame([[adjusted_rand_score(labels[a], labels[b]) for b in keys] for a in keys], index=keys, columns=keys)
ari.to_csv(TAB / "08_resolution_pairwise_ari.csv")
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(ari, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(keys)), keys, rotation=60, ha="right")
ax.set_yticks(range(len(keys)), keys)
for i in range(len(keys)):
    for j in range(len(keys)): ax.text(j, i, f"{ari.iloc[i,j]:.2f}", ha="center", va="center", color="white" if ari.iloc[i,j] < .5 else "black")
fig.colorbar(im, ax=ax, label="Adjusted Rand Index")
ax.set_title("Resolution sensitivity (shared expression graph)")
fig.tight_layout(); fig.savefig(FIG / "08_resolution_ari.png", dpi=170); plt.show()

## 5. Spatial continuity: within-cluster edge fraction

Evaluate the fraction of spatial edges joining bins with identical **original** Leiden labels. Compare to a shuffled-label reference (100 permutations). This reference preserves cluster sizes but **not** spatial dependence; it is a descriptive baseline, not a calibrated inferential p-value. The spatial adjacency is treated as undirected and self-loops are excluded.

In [ ]:
W = adata.obsp["spatial_connectivities"].tocsr()
W = W.maximum(W.T).tocsr()
W.setdiag(0); W.eliminate_zeros()
r, col = sparse.triu(W, k=1).nonzero()
assert len(r) > 0, "No spatial edges found"
code = original_leiden.cat.codes.to_numpy()
observed = float(np.mean(code[r] == code[col]))
N_PERMS = 100
null = np.empty(N_PERMS, dtype=float)
for i in range(N_PERMS):
    shuffled = rng.permutation(code)
    null[i] = np.mean(shuffled[r] == shuffled[col])
spatial_summary = pd.DataFrame([{"n_undirected_edges": len(r), "within_cluster_fraction": observed,
    "shuffled_mean": float(null.mean()), "shuffled_sd": float(null.std(ddof=1)),
    "n_shuffles": N_PERMS, "interpretation": "descriptive; shuffled labels ignore spatial dependence"}])
spatial_summary.to_csv(TAB / "08_spatial_continuity.csv", index=False)
print(spatial_summary.to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(null, bins=20, alpha=.7, label="Shuffled labels")
ax.axvline(observed, color="crimson", lw=2, label="Observed")
ax.set(xlabel="Fraction of same-cluster spatial edges", ylabel="Count", title="Spatial continuity: descriptive shuffled-label reference")
ax.legend(); fig.tight_layout(); fig.savefig(FIG / "08_spatial_continuity.png", dpi=170); plt.show()

## 6. Exploratory QC threshold sensitivity (without filtering)

Calculate **how many bins would pass** example thresholds; do not subset, recluster, or treat these thresholds as validated. Report retained fraction per original cluster to flag possible differential impact. Thresholds are illustrative and must be reviewed with QC distributions and tissue context.

In [ ]:
MIN_COUNTS = 100
MIN_GENES = 50
MAX_PCT_MT = 25.0
mask = (obs["total_counts"].to_numpy() >= MIN_COUNTS) & (obs["n_genes_by_counts"].to_numpy() >= MIN_GENES)
mt_used = "pct_counts_mt" in qc_cols
if mt_used:
    mt = obs["pct_counts_mt"].to_numpy(dtype=float)
    mask &= np.isfinite(mt) & (mt <= MAX_PCT_MT)
retention = []
for cl in clusters:
    idx = obs["leiden"].to_numpy() == cl
    retention.append({"leiden": cl, "n_bins": int(idx.sum()), "would_retain": int(mask[idx].sum()), "would_retain_fraction": float(mask[idx].mean())})
retention_df = pd.DataFrame(retention)
retention_df.to_csv(TAB / "08_hypothetical_qc_retention.csv", index=False)
print("Illustrative thresholds:", {"min_counts": MIN_COUNTS, "min_genes": MIN_GENES, "max_pct_mt": MAX_PCT_MT if mt_used else "not available"})
print("Would retain:", int(mask.sum()), "/", len(mask), "bins. NO FILTERING APPLIED.")
print(retention_df.sort_values("would_retain_fraction").head(10).to_string(index=False))
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(retention_df["leiden"], retention_df["would_retain_fraction"])
ax.set(xlabel="Leiden cluster", ylabel="Hypothetical retention fraction", ylim=(0, 1.02), title="Illustrative QC thresholds — not applied")
ax.tick_params(axis="x", rotation=90)
fig.tight_layout(); fig.savefig(FIG / "08_hypothetical_qc_retention.png", dpi=170); plt.show()

## 7. Save and reload validation

Keep all original labels and analysis products, adding only resolution-sensitivity labels and diagnostic metadata. This notebook does **not** confirm any biological cell-type annotations.

In [ ]:
assert adata.obs["leiden"].equals(original_leiden), "Original cluster labels changed"
adata.uns["step08_settings"] = {
    "diagnostic_only": True,
    "qc_filters_applied": False,
    "resolution_values": RESOLUTIONS,
    "seed": SEED,
    "spatial_shuffles": N_PERMS,
    "spatial_null_calibrated": False,
    "image_registration_verified": False,
}
adata.write_h5ad(OUTPUT, compression="gzip")
check = sc.read_h5ad(OUTPUT)
assert check.shape == adata.shape
assert check.obs["leiden"].equals(original_leiden)
assert np.array_equal(check.obsm["spatial"], adata.obsm["spatial"])
assert all(k in check.obsp for k in ("connectivities", "distances", "spatial_connectivities", "spatial_distances"))
assert "counts" in check.layers and sparse.issparse(check.layers["counts"])
assert "provisional_domain_program" in check.obs
assert "step08_settings" in check.uns
assert (TAB / "08_resolution_sensitivity.csv").exists()
assert (FIG / "08_resolution_ari.png").exists()
print("PASS — 08_robustness_checks completed")
print("AnnData:", check.shape)
print("Original Leiden clusters:", check.obs["leiden"].nunique())
print("Resolution variants:", len(RESOLUTIONS))
print("Output:", OUTPUT.relative_to(ROOT))

### Next

Review QC effects, spatial continuity, and resolution sensitivity **before deciding whether to rerun upstream steps with QC filtering**. Then create a reproducible pipeline runner, README, and finally the pinned `environment.yml`. Tangram remains optional and requires a biologically suitable matched scRNA-seq reference.